***EDA Kioscos***

Hay un total 753 registros.

Hay nulos totales en el campo `City`. Existen otros campos en nulo, se le asignará 'No tiene'.

Cada comercio no tiene ni barrio ni comuna.

Se filtrará por los campos Latitude y Longitude que tengan un valor no nulo y se ubique dentro de CABA. 

Se agregará el barrio y comuna buscando en el dataset de barrios por los campos `Latitude` y `Longitude`. Se filtrará tambien por barrios no nulos.

Se filtrará tambien por `Category` que sea Kiosco | Quiosco | Tienda de Golosinas

Se filtrará por valores únicos de Name - Latitude - Longitude. 

Limpiar caracteres raros como 


In [0]:
%sql

DESCRIBE proyecto_final.raw.kioscos_bronze;

In [0]:
%sql

select * from proyecto_final.raw.kioscos_bronze
limit 20;

In [0]:
%sql
select count(*) as total_registros,
count(*)-count(City) as city_nulos,
count(*)-count(Name) as name_nulos,
count(*)-count(Category) as category_nulos,
count(*)-count(Address) as address_nulos,
count(*)-count(Phone) as phone_nulos,
count(*)-count(Website) as website_nulos,
count(*)-count(Email) as email_nulos,
count(*)-count(Instagram) as instagram_nulos,
count(*)-count(Facebook) as facebook_nulos,
count(*)-count(Rating) as rating_nulos,
count(*)-count(Reviews_count) as reviews_count_nulos,
count(*)-count(Claimed) as claimed_nulos,
count(*)-count(Attributes) as attributes_nulos,
count(*)-count(Top_Review) as top_review_nulos,
count(*)-count(Working_Hours) as working_hours_nulos,
count(*)-count(Image_URL) as image_url_nulos,
count(*)-count(Latitude) as latitude_nulos,
count(*)-count(Longitude) as longitude_nulos,
count(*)-count(Google_URL) as google_url_nulos
from proyecto_final.raw.kioscos_bronze

-- city rating reviews top working hours image

-- latitud y longitud no deben ser nulos

In [0]:
%sql
-- Duplicados
select 
Name,
latitude,
longitude,
count(*) as cantidad
from proyecto_final.raw.kioscos_bronze
group by Name,latitude, longitude
having count(*) > 1
order by cantidad desc

In [0]:
%sql
-- Hacer consulta de duplicados por grupo

In [0]:
%sql
select * from proyecto_final.raw.kioscos_bronze
WHERE Latitude > -34.50 OR Latitude < -34.70 
   OR Longitude > -58.30 OR Longitude < -58.55;

In [0]:
select 
Category,
count(*) as cantidad
from proyecto_final.raw.kioscos_bronze
group by Category
order by cantidad desc

In [0]:
Select distinct Name from proyecto_final.raw.kioscos_bronze
where Category RLIKE 'Quiosco|Kiosco|Tienda de Golosinas'
and not( Latitude > -34.50 OR Latitude < -34.70 
   OR Longitude > -58.30 OR Longitude < -58.55)

In [0]:
%sql


-- Asigno barrio y comuna segun la latitud y longitud 
SELECT 
    k.*, 
    b.nombre AS barrio_normalizado,
    b.comuna AS comuna_normalizada
FROM (
    -- Para los kioscos usamos ST_Point directamente con sus grados
    SELECT *, 
           st_point(Longitude, Latitude) as geom_punto
    FROM proyecto_final.raw.kioscos_bronze
) k
LEFT JOIN proyecto_final.raw.barrios_bronze b 
  -- Verificamos si el punto del kiosco cae dentro del polígono del barrio
  ON ST_Contains(st_geomfromwkt(b.geometry), k.geom_punto)
where b.nombre is NOT null;

In [0]:
%sql

CREATE OR REPLACE VIEW v_kioscos_limpieza AS
WITH kioscos_procesados AS (
    SELECT 
        -- 1. Deduplicación: Numerar registros por Nombre, Latitud y Longitud
        ROW_NUMBER() OVER(
            PARTITION BY 
                LOWER(REPLACE(REPLACE(TRIM(name), '"', ''), '', '')), 
                latitude, 
                longitude 
            ORDER BY fecha_ingesta DESC
        ) AS rn,
        
        -- 2. Limpieza de texto, caracteres raros y asignación de 'No tiene' a nulos
        COALESCE(LOWER(REPLACE(REPLACE(TRIM(name), '"', ''), '', '')), 'No tiene') AS nombre,
        COALESCE(LOWER(REPLACE(REPLACE(TRIM(category), '"', ''), '', '')), 'No tiene') AS categoria,
        COALESCE(LOWER(REPLACE(REPLACE(TRIM(address), '"', ''), '', '')), 'No tiene') AS direccion,
        
        -- Datos de contacto y redes sociales
        COALESCE(trim(REPLACE(REPLACE(REPLACE(TRIM(phone), '"', ''), '', ''), '', '')), 'No tiene') AS telefono,
        COALESCE(REPLACE(REPLACE(TRIM(website), '"', ''), '', ''), 'No tiene') AS sitio_web,
        COALESCE(LOWER(REPLACE(REPLACE(TRIM(email), '"', ''), '', '')), 'No tiene') AS email,
        COALESCE(LOWER(REPLACE(REPLACE(TRIM(instagram), '"', ''), '', '')), 'No tiene') AS instagram,
        COALESCE(LOWER(REPLACE(REPLACE(TRIM(facebook), '"', ''), '', '')), 'No tiene') AS facebook,
        
        -- Metadatos y reseñas
        COALESCE(rating, 'No tiene') AS rating,
        COALESCE(reviews_count, 'No tiene') AS reviews_count,
        CAST(claimed AS BOOLEAN) AS claimed,
        COALESCE(LOWER(REPLACE(REPLACE(TRIM(top_Review), '"', ''), '', '')), 'No tiene') AS top_review,
        COALESCE(LOWER(REPLACE(REPLACE(TRIM(working_Hours), '"', ''), '', '')), 'No tiene') AS horario,
        
        -- Coordenadas
        latitude,
        longitude,
        fecha_ingesta

    FROM proyecto_final.raw.kioscos_bronze
    
    -- 3. Filtros iniciales de coordenadas y categoría
    WHERE latitude IS NOT NULL 
      AND longitude IS NOT NULL
      AND LOWER(REPLACE(REPLACE(TRIM(category), '"', ''), '', '')) IN ('kiosco', 'quiosco', 'tienda de golosinas')
)

SELECT 
    k.nombre,
    k.categoria,
    
    -- 4. Asignación de Barrio y Comuna mediante cruce espacial
    b.barrio_nombre AS barrio,
    b.comuna_id AS comuna,
    
    k.direccion,
    k.telefono,
    k.sitio_web,
    k.email,
    k.instagram,
    k.facebook,
    k.rating,
    k.reviews_count,
    k.claimed,
    k.top_review,
    k.horario,
    k.latitude,
    k.longitude,
    k.fecha_ingesta

FROM kioscos_procesados k
-- Cruce espacial: Se verifica si el punto del kiosco está dentro del polígono del barrio
LEFT JOIN v_barrios_limpieza b 
  ON st_contains(st_geomfromwkt(b.coordenadas), ST_Point(k.longitude, k.latitude))

-- 5. Filtros finales: Mantener solo registros únicos y dentro de CABA
WHERE k.rn = 1 
  AND b.barrio_nombre IS NOT NULL;

In [0]:
select * from v_kioscos_limpieza